# CSCI E-89 Deep Learning — Assignment 03

**Name:** Jazmyn Stokes
**Problem 1 (25%):** Use Claude Code to incrementally build a PyTorch image classifier
for Fashion MNIST (the "Building an Image Classifier with PyTorch" section of
`10_neural_nets_with_pytorch.ipynb`, Chapter 10 of *Hands-On Machine Learning with
Scikit-Learn and PyTorch*, Aurelien Geron, 2025), adding a training-accuracy plot.

Each numbered section below corresponds to one incremental prompt given to Claude
Code in this session, in the order it was requested. Cells all run top to bottom and
share one namespace, so later cells can reuse variables/functions defined earlier.


## Prompt log

| # | Prompt to Claude Code | What it produced |
|---|---|---|
| 1 | Setup: imports, pick best device, seed 42, matplotlib defaults, print torch version + device | Imports / device / seed / plot defaults (below) |
| 2 | Load Fashion MNIST with TorchVision, convert to [0,1] float tensors, re-seed 42, split 55k/5k train/validation, print split sizes + class names | Dataset loading + split (below) |
| 3 | Wrap the three splits in DataLoaders (batch_size=32, shuffle train only), print one sample's shape/dtype/label | DataLoaders + sample check (below) |
| 4 | Define the `ImageClassifier` MLP (784/300/100/10), move it to device, create the loss, print the model + parameter count | Model + loss (below) |
| 5 | Define `evaluate_tm()` and `train2()` (returns a history dict), no training run yet | Training/eval helper functions (below) |
| 6 | Train for 20 epochs with SGD(lr=0.1) + torchmetrics multiclass accuracy, keep the returned history | Training run (below) |
| 7 | Plot training vs. validation accuracy per epoch (required addition) + a second figure for training loss | Accuracy/loss plots (below) |
| 8 | Predict 3 validation images, softmax + top-4 probabilities (mps->cpu workaround for round()), show the images | Prediction + visualization (below) |
| 9 | Final evaluation on the held-out test set, print test accuracy + parameter count | Test evaluation (below) |


---
## 1. Setup — imports, device selection, seed, plotting defaults

**Prompt:** "imports (numpy, torch, nn, F, torchmetrics, matplotlib), pick the best
device (Mac), seed 42, set some matplotlib defaults, and print the torch version and
device."

- Imports cover tensors/arrays (`numpy`, `torch`), building blocks for the model
  (`torch.nn` as `nn`, `torch.nn.functional` as `F`), metric tracking (`torchmetrics`),
  and plotting (`matplotlib.pyplot`).
- Device selection checks CUDA first (a discrete/cloud GPU), then Apple Silicon's
  Metal backend (`mps`, what a Mac with an M-series chip uses), then falls back to
  `cpu` everywhere else — this makes the notebook portable across machines.
- `torch.manual_seed(42)` (plus `np.random.seed(42)` for any NumPy-side randomness)
  is set once, up front, so later steps such as the train/validation split, weight
  initialization, and DataLoader shuffling are reproducible across reruns.
- Matplotlib rc defaults set a consistent, readable font/label/legend size for every
  figure drawn later in the notebook (e.g. the training-accuracy plot in a later
  section), instead of repeating `plt.rc(...)` calls near each plot.


In [ ]:
# Step 1 — Setup: imports, device selection, reproducibility seed, plot defaults.
# Everything downstream (data loading, model, training loop, plots) depends on the
# `device` variable and the fixed seed defined here, and this cell must be run first.

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import matplotlib.pyplot as plt

# Pick the fastest device available on this machine.
# Order matters: CUDA (NVIDIA GPU) first, then Apple Silicon's "mps" backend
# (what this Mac uses), then CPU as the universal fallback so the notebook still
# runs anywhere.
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Fix the random seed once, up front, so the dataset split, weight initialization,
# and DataLoader shuffling are all reproducible on every rerun of this notebook.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Matplotlib defaults applied notebook-wide, so every later figure (e.g. the
# training/validation accuracy curves) is consistently sized and legible without
# repeating these settings near each plot.
plt.rc("font", size=12)
plt.rc("axes", labelsize=12, titlesize=14)
plt.rc("legend", fontsize=12)
plt.rc("figure", figsize=(8, 5))

print(f"PyTorch version : {torch.__version__}")
print(f"Selected device : {device}")


### Sanity check

A quick check that every import resolved, the device object actually accepts a
tensor, and the seed produces the same "random" values on repeated calls in this
run — confirming the setup cell above is solid before anything is built on top of
it.


In [ ]:
# Sanity check for the setup cell: confirms torchmetrics imported correctly,
# a tensor can actually be moved to the selected device, and the seed is doing its
# job (two manual_seed(42) calls in a row must reproduce the same values).
assert isinstance(device, str) and device in {"cuda", "mps", "cpu"}

_probe = torch.randn(3, 3).to(device)
assert _probe.device.type == device
print(f"Tensor successfully placed on '{device}', shape {tuple(_probe.shape)}")

torch.manual_seed(SEED)
_a = torch.rand(5)
torch.manual_seed(SEED)
_b = torch.rand(5)
assert torch.equal(_a, _b), "Seeding is not reproducible!"
print("Seed reproducibility check passed.")

# torchmetrics import check: build a trivial metric object (no data yet).
_metric_check = torchmetrics.Accuracy(task="multiclass", num_classes=10)
print(f"torchmetrics OK: {_metric_check}")

del _probe, _a, _b, _metric_check  # scratch variables only, not needed later


---
## 2. Load Fashion MNIST, transform, and split into train / validation

**Prompt:** "load Fashion MNIST from torchvision into a datasets/ folder, convert to
float tensors scaled to [0,1] with transforms.v2. Re-seed to 42, then split the 60k
training images into 55k train / 5k validation. Print the split sizes and the class
names."

- `torchvision.datasets.FashionMNIST` downloads (or reuses, if already cached) the
  dataset into a local `datasets/` folder. `train=True` gives the 60,000-image
  training pool; `train=False` gives the 10,000-image held-out test set, which is
  loaded here but not touched again until final evaluation.
- The `transforms.v2` pipeline (`ToImage()` then `ToDtype(torch.float32, scale=True)`)
  turns each raw PIL/uint8 image into a `[1, 28, 28]` float32 tensor with pixel
  values scaled from `[0, 255]` down to `[0, 1]`. Unscaled 0-255 inputs would make
  gradient descent unstable at the learning rate used later.
- `torch.manual_seed(42)` is called again immediately before the split, so the exact
  same 55,000/5,000 partition is drawn no matter which earlier cells were or weren't
  re-run first (this is a separate, explicit re-seed as the prompt asked for, on top
  of the one already set in the setup cell).
- `torch.utils.data.random_split` divides the 60,000 training images into 55,000 for
  training and 5,000 held out for validation during training.
- The dataset's `.classes` attribute gives the 10 human-readable Fashion MNIST
  category names (e.g. "T-shirt/top", "Sneaker"), which later cells use to label
  predictions.


In [ ]:
# Step 2 — Load Fashion MNIST, scale to [0, 1], re-seed, split 55k/5k train/validation.
# Depends on `torch`, `SEED` from the setup cell above.

import torchvision
import torchvision.transforms.v2 as T

# ToImage() gives a tensor in [C, H, W] layout; ToDtype(..., scale=True) casts the
# uint8 pixels to float32 and divides by 255, putting every input in [0, 1].
to_float_tensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, download=True, transform=to_float_tensor)
test_data = torchvision.datasets.FashionMNIST(
    root="datasets", train=False, download=True, transform=to_float_tensor)

# Human-readable label names, in class-index order (0-9).
class_names = train_and_valid_data.classes

# Re-seed right before the split (as requested) so the 55k/5k partition is
# reproducible regardless of what ran earlier in the session.
torch.manual_seed(SEED)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [55000, 5000])

print(f"train      : {len(train_data):,} images")
print(f"validation : {len(valid_data):,} images")
print(f"test       : {len(test_data):,} images")
print(f"class names: {class_names}")


---
## 3. DataLoaders for train / validation / test

**Prompt:** "DataLoaders for all three splits, batch_size=32, shuffle the training
one only. Then print one sample's shape, dtype and label so I can see what's going
in."

- `DataLoader` wraps each split so training/evaluation can iterate over mini-batches
  instead of single images. `batch_size=32` matches the assignment's template
  pipeline.
- `shuffle=True` only on the training loader: reshuffling each epoch decorrelates
  consecutive mini-batches, which helps SGD converge. Validation and test are always
  evaluated over the full set regardless of order, so shuffling them would have no
  benefit and only adds overhead.
- Printing one raw sample (straight from `train_data`, before batching) confirms the
  transform from Step 2 produced the expected `[1, 28, 28]` float32 tensor scaled to
  `[0, 1]`, and that the label lines up with a valid `class_names` entry, before any
  model sees the data.


In [ ]:
# Step 3 — DataLoaders for train/validation/test, then inspect one sample.
# Depends on `train_data`, `valid_data`, `test_data`, `class_names`, `SEED` from
# Step 2.

from torch.utils.data import DataLoader

torch.manual_seed(SEED)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

# Each dataset entry is an (image, label) tuple; look at the very first one.
X_sample, y_sample = train_data[0]
print("sample image shape :", tuple(X_sample.shape))   # [channels, rows, cols]
print("sample image dtype :", X_sample.dtype)
print("sample label       :", y_sample, "->", class_names[y_sample])


---
## 4. The model: `ImageClassifier` (MLP) and the loss function

**Prompt:** "an nn.Module called ImageClassifier: Flatten, Linear, ReLU, Linear,
ReLU, Linear, no activation at the end since CrossEntropyLoss wants logits. Build it
with 784/300/100/10, move it to device, make the loss, and print the model plus the
parameter count."

- `ImageClassifier` subclasses `nn.Module` and defines the forward pass as an
  `nn.Sequential` stack: `Flatten` turns each `[1, 28, 28]` image into a 784-length
  vector, then two hidden `Linear + ReLU` blocks (784→300→100), then a final
  `Linear(100, 10)` output layer with **no activation function**.
- No activation on the output layer is deliberate: `nn.CrossEntropyLoss` applies
  `log_softmax` internally and expects raw, unnormalized logits as input. Adding a
  softmax here would apply it twice and slow down learning.
- The model is moved to `device` (selected in Step 1) with `.to(device)` so its
  parameters live on the same device the input batches will be moved to during
  training/evaluation — a device mismatch between model and data raises an error.
- `nn.CrossEntropyLoss()` is the standard multi-class classification loss, combining
  log-softmax and negative log-likelihood in one call.
- Printing the model shows the layer stack; the parameter count (784*300 + 300 +
  300*100 + 100 + 100*10 + 10 = 266,610) is a sanity check against the assignment's
  reference architecture.


In [ ]:
# Step 4 — Define the ImageClassifier MLP, move it to device, create the loss.
# Depends on `nn`, `device` from Step 1.

class ImageClassifier(nn.Module):
    """Fully connected classifier for 28x28 grayscale Fashion MNIST images.

    Architecture: Flatten -> Linear(784,300) -> ReLU -> Linear(300,100) -> ReLU
    -> Linear(100,10). The final layer outputs raw logits (no activation), since
    nn.CrossEntropyLoss expects logits and applies log-softmax internally.
    """

    def __init__(self, n_inputs=784, n_hidden1=300, n_hidden2=100, n_classes=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),                          # [N, 1, 28, 28] -> [N, 784]
            nn.Linear(n_inputs, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes),        # raw logits, no activation
        )

    def forward(self, X):
        return self.mlp(X)


torch.manual_seed(SEED)  # reproducible weight initialization
model = ImageClassifier(784, 300, 100, 10).to(device)

# Standard multi-class classification loss: expects raw logits + integer class
# labels, and applies log-softmax internally.
criterion = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\ntotal trainable parameters: {n_params:,}")


---
## 5. Training and evaluation helper functions

**Prompt:** "just the functions, don't run anything yet. An evaluate_tm() that runs
a loader under no_grad and returns the metric, and a train2() that trains and
returns a history dict with train_losses, train_metrics and valid_metrics. I need
that history returned, not just printed — I'm plotting it later. Print the metrics
each epoch too."

Only the two function definitions are added in this cell — nothing is executed or
trained yet (that's the next step, once an optimizer exists).

- `evaluate_tm(model, data_loader, metric)` switches the model to `eval()` mode
  (disables dropout / uses running batch-norm stats — not used by this MLP, but it's
  the correct habit), resets the `torchmetrics` metric so no stale state leaks in
  from a previous call, then loops over the loader under `torch.no_grad()` (no
  autograd graph needed for evaluation, which saves memory and time) accumulating
  predictions into the metric. It returns the aggregated metric value via
  `metric.compute()`.
- `train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs)`
  runs the standard training loop for `n_epochs`: for each batch, zero the
  gradients, forward pass, compute loss, backward pass, optimizer step, and
  accumulate both the running loss and the training metric. At the end of each
  epoch it computes the epoch's average training loss and training metric, then
  calls `evaluate_tm` on `valid_loader` to get the validation metric, prints all
  three, and appends them to a `history` dict with keys `train_losses`,
  `train_metrics`, and `valid_metrics`. Returning `history` (rather than only
  printing) is what makes the training-accuracy plot in a later step possible.


In [ ]:
# Step 5 — Training/evaluation helper functions only. Nothing is executed here;
# no optimizer or training run exists yet, so this cell just defines the two
# functions used by the next step.

def evaluate_tm(model, data_loader, metric):
    """Run `model` over every batch in `data_loader` under no_grad and return the
    aggregated value of `metric` (a torchmetrics metric object)."""
    model.eval()           # eval mode: disables dropout / uses running BN stats
    metric.reset()         # torchmetrics accumulates internally; clear stale state
    with torch.no_grad():  # no autograd graph needed for evaluation
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()


def train2(model, optimizer, criterion, metric, train_loader, valid_loader, n_epochs):
    """Train `model` for `n_epochs`, printing loss/accuracy each epoch, and return
    a history dict with keys 'train_losses', 'train_metrics', 'valid_metrics'
    (one entry per epoch) so the caller can plot learning curves afterward."""
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}

    for epoch in range(n_epochs):
        model.train()
        metric.reset()
        running_loss, n_seen = 0.0, 0

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            n_seen += X_batch.size(0)
            metric.update(y_pred, y_batch)

        train_loss = running_loss / n_seen
        train_metric = metric.compute().item()
        valid_metric = evaluate_tm(model, valid_loader, metric).item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)
        history["valid_metrics"].append(valid_metric)

        print(f"epoch {epoch + 1:2d}/{n_epochs} | "
              f"train loss: {train_loss:.4f} | "
              f"train acc: {train_metric:.4f} | "
              f"valid acc: {valid_metric:.4f}")

    return history


print("evaluate_tm() and train2() defined. Nothing executed yet.")


---
## 6. Run training — 20 epochs, SGD(lr=0.1)

**Prompt:** "train for 20 epochs with SGD lr=0.1 and a torchmetrics multiclass
accuracy on the same device. Keep the history."

- `torch.optim.SGD(model.parameters(), lr=0.1)` is plain (non-momentum) stochastic
  gradient descent at a learning rate of 0.1, matching the assignment's reference
  pipeline.
- `torchmetrics.Accuracy(task="multiclass", num_classes=10)` computes classification
  accuracy across the 10 Fashion MNIST classes. It's moved to `device` with
  `.to(device)` — a torchmetrics metric must live on the same device as the tensors
  it's updated with, or it raises a device-mismatch error.
- `train2(...)` (defined in Step 5) is called with `n_epochs=20`; its return value is
  kept in `history` — this is the actual, real training run, and this is the one
  long-running cell in the notebook. On a Mac with Apple Silicon (`mps`) or a GPU,
  20 epochs typically takes a few minutes; longer on CPU.
- `history` is what the next step plots, so nothing here should be re-run without
  also expecting the training-accuracy plot to change.


In [ ]:
# Step 6 — Train for 20 epochs with SGD(lr=0.1); keep the returned history.
# Depends on `model`, `criterion`, `device` (Step 4), `train_loader`, `valid_loader`
# (Step 3), and `train2`/`evaluate_tm` (Step 5). This is the long-running cell —
# expect it to take a few minutes.

n_epochs = 20

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# The metric must live on the same device as the tensors it consumes, otherwise
# torchmetrics raises a device-mismatch error.
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

history = train2(model, optimizer, criterion, accuracy,
                  train_loader, valid_loader, n_epochs)


---
## 7. Plot training accuracy (required) and training loss

**Prompt:** "add code that will plot the training accuracy per epoch, with
validation accuracy on the same plot. Put training at the epoch midpoint since it's
an average over the epoch. Labels, grid, legend, y range 0.7-1.0. Print the final
numbers, and add a second figure for the training loss."

This is the one addition Assignment 03 explicitly requires on top of the book's
reference pipeline.

- Training accuracy is an average computed *over the course of* each epoch (it
  accumulates as each mini-batch is seen), so it's plotted at `epoch + 0.5` — the
  epoch's midpoint — to reflect that it's a running average, not a single point-in-
  time measurement.
- Validation accuracy, in contrast, is measured once, in full, *after* the epoch
  finishes, so it's plotted at `epoch + 1.0` — the epoch's end.
- Labels, a grid, and a legend are added so the plot is self-explanatory; the y-axis
  is fixed to `[0.7, 1.0]` so run-to-run comparisons share the same scale and small
  differences near the top of the range remain visible.
- The final training and validation accuracy values are printed numerically under
  the plot, not just shown visually.
- A second figure plots training loss per epoch as a cross-check: loss falling while
  accuracy rises is the expected, healthy pattern; if they diverge, something in the
  training loop deserves a second look.


In [ ]:
# Step 7 — Plot training vs. validation accuracy (the assignment's required
# addition), print the final numbers, and plot training loss as a cross-check.
# Depends on `history`, `n_epochs` from Step 6.

epochs = np.arange(n_epochs)

# Training accuracy is a running average over the epoch, so it belongs at the
# epoch's midpoint (epoch + 0.5). Validation accuracy is measured once, at the
# end of the epoch, so it sits at epoch + 1.0.
plt.figure()
plt.plot(epochs + 0.5, history["train_metrics"], ".--", label="Training accuracy")
plt.plot(epochs + 1.0, history["valid_metrics"], ".-", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fashion MNIST classifier - learning curves")
plt.grid(True)
plt.legend()
plt.axis([0.5, n_epochs, 0.7, 1.0])
plt.show()

print(f"final training accuracy   : {history['train_metrics'][-1]:.4f}")
print(f"final validation accuracy : {history['valid_metrics'][-1]:.4f}")

# Second figure: training loss per epoch, as a cross-check. Accuracy rising while
# loss falls is the expected pattern for a healthy training run.
plt.figure()
plt.plot(epochs + 0.5, history["train_losses"], ".--", color="tab:red",
         label="Training loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Fashion MNIST classifier - training loss")
plt.grid(True)
plt.legend()
plt.show()

print(f"final training loss       : {history['train_losses'][-1]:.4f}")


---
## 8. Predict on 3 validation images and visualize

**Prompt:** "predict on 3 images from the validation loader, print predicted vs
actual class names. Then softmax the logits and show the probabilities rounded,
plus the top 4 per image. Heads up: round() isn't implemented on mps, so move to
cpu first. Then show the 3 images with their labels."

- The model is switched to `eval()` mode and one batch is pulled from
  `valid_loader`; only the first 3 images/labels of that batch are used.
- Raw logits from the model are converted to predicted class indices with
  `argmax(dim=1)`, then mapped back to human-readable names via `class_names`, and
  printed alongside the true labels for a quick correct/incorrect check.
- `F.softmax(logits, dim=1)` turns the logits into per-class probabilities.
  **`Tensor.round(decimals=...)` is not implemented on Apple Silicon's `mps`
  backend**, so whenever `device == "mps"` the tensor is moved to `cpu` first with
  `.cpu()` before rounding — this keeps the notebook from crashing on a Mac while
  still working unchanged on `cuda`/`cpu`.
- `torch.topk(logits, k=4, dim=1)` finds each image's 4 highest-scoring classes;
  those top-4 logits are re-softmaxed on their own (`F.softmax` over just the 4
  values) so the reported top-4 probabilities sum to 1 for easy reading, rather than
  showing raw slices of the full 10-way distribution.
- Finally, the 3 images are displayed with `imshow` (moved to `cpu` for plotting,
  since matplotlib only works with CPU/NumPy arrays), each titled with its predicted
  and true class name.


In [ ]:
# Step 8 — Predict on 3 validation images, show softmax probabilities + top-4,
# then display the images with their predicted/true labels.
# Depends on `model`, `device`, `valid_loader`, `class_names` from earlier steps.

model.eval()

# Take the first batch from the validation loader and predict on its first 3
# images.
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
y_new = y_new[:3]

with torch.no_grad():
    y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)  # index of the largest logit per image

print("predicted :", [class_names[i] for i in y_pred])
print("actual    :", [class_names[i] for i in y_new])
print("correct   :", (y_pred.cpu() == y_new).tolist())

# The model outputs logits; softmax turns them into class probabilities.
y_proba = F.softmax(y_pred_logits, dim=1)

# round(decimals=...) is not implemented on the "mps" backend, so move to cpu
# first whenever that's the active device (a no-op cost on cuda/cpu).
if device == "mps":
    y_proba = y_proba.cpu()

print("\nclass probabilities (rounded):")
print(y_proba.round(decimals=3))

# Top-4 classes per image, re-softmaxed over just those 4 logits so the reported
# values sum to 1 per image.
y_top4_values, y_top4_indices = torch.topk(y_pred_logits, k=4, dim=1)
y_top4_probas = F.softmax(y_top4_values, dim=1)
if device == "mps":
    y_top4_probas = y_top4_probas.cpu()

print("\ntop-4 probabilities:")
print(y_top4_probas.round(decimals=3))
print("\ntop-4 class indices:")
print(y_top4_indices)


Show the 3 images alongside their predicted and true labels, so the numbers above
can be checked visually against the actual Fashion MNIST items.


In [ ]:
# Display the 3 validation images with their predicted vs. true class names.
fig, axes = plt.subplots(1, 3, figsize=(9, 3.4))
for ax, image, pred, true in zip(axes, X_new.cpu(), y_pred.cpu(), y_new):
    ax.imshow(image.squeeze(), cmap="binary")  # squeeze drops the channel axis
    ax.set_title(f"pred: {class_names[pred]}\ntrue: {class_names[true]}",
                 fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()


---
## 9. Final evaluation on the test set

**Prompt:** "one final run on the test set, print the test accuracy and the
parameter count."

- `test_loader` (built in Step 3) has not been touched anywhere in training or
  validation — it's used here for the first and only time, which is what makes this
  a fair, held-out estimate of how the model generalizes.
- Reuses `evaluate_tm` (Step 5) with the same `accuracy` metric object (Step 6);
  `evaluate_tm` resets the metric internally before accumulating over the test set,
  so no state leaks in from the training/validation runs.
- The parameter count is recomputed the same way as in Step 4, as a final summary
  next to the accuracy number.


In [ ]:
# Step 9 — Final, single evaluation on the held-out test set.
# Depends on `model`, `test_loader`, `accuracy` from earlier steps. This is the
# only place test_loader is used.

test_accuracy = evaluate_tm(model, test_loader, accuracy).item()
n_params_final = sum(p.numel() for p in model.parameters())

print(f"test accuracy      : {test_accuracy:.4f}")
print(f"model parameters   : {n_params_final:,}")
